# 악보 판독 평가 — pilot20 (실습실 RTX 4090용)

Qwen3-VL이 악보 이미지를 읽는지(음자리표, 음높이, 음표/쉼표 종류, 임시표, 조표, 박자표, 장식음)를 **평가만** 합니다.

> **pilot20 데이터는 평가 전용입니다. 학습, 파인튜닝, few-shot 예시, 프롬프트 튜닝에 쓰지 않습니다.**
> 이 노트북은 모델 가중치를 바꾸지 않고, 정답(`answer`)은 채점 단계에서만 읽습니다. 모델에는 이미지와 질문만 전달합니다.

- 데이터: `data/pilot20/pilot_test.jsonl` (20문항, 이미지 `data/pilot20/images/*.png`)
- 모델: `Qwen/Qwen3-VL-4B-Instruct` (vLLM 서버), greedy(temperature 0)
- 채점: 규칙 기반(`scoring.py`), judge 모델 없음. **strict**(정규화 후 완전 일치) / **lenient**(문장형 답변 허용) 두 가지
- 결과: `results/pilot20/` (git에 올리지 않음)

## 1. 설정

In [ ]:
import os, sys, json, time, signal, subprocess, base64
from pathlib import Path

HERE = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'scoring.py').exists() and (p / 'data' / 'pilot20').exists())
sys.path.insert(0, str(HERE))
import scoring

MODEL     = 'Qwen/Qwen3-VL-4B-Instruct'
REVISION  = 'ebb281ec70b05090aa6165b016eac8ec08e71b17'   # baseline과 같은 커밋
DATA_DIR  = HERE / 'data' / 'pilot20'
OUT_DIR   = HERE / 'results' / 'pilot20'
OUT_DIR.mkdir(parents=True, exist_ok=True)

PORT          = 8000
MAX_MODEL_LEN = 8192
GPU_MEM_UTIL  = 0.90
MAX_NEW_TOKENS = 512            # 직접 답하기. CoT로 바꾸면 4096 이상 권장
USE_COT       = False           # True면 '단계적으로 생각한 뒤 Answer: 로 마무리' 프롬프트
MIN_PIXELS, MAX_PIXELS = 1_003_520, 4_014_080            # baseline(Run B)과 같은 이미지 해상도 설정 (1280~5120 x 28^2)
CONCURRENCY   = 8

rows = [json.loads(l) for l in open(DATA_DIR / 'pilot_test.jsonl', encoding='utf-8')]
assert len(rows) == 20 and all((DATA_DIR / r['image']).exists() for r in rows)
print('문항 수:', len(rows), '| 카테고리:', sorted({r['category'] for r in rows}))
print(subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,memory.used', '--format=csv'], capture_output=True, text=True).stdout)

## 2. vLLM 서버 실행

In [ ]:
import requests

def stop_server(proc):
    try:
        os.killpg(proc.pid, signal.SIGTERM); proc.wait(timeout=120)
    except Exception:
        try: os.killpg(proc.pid, signal.SIGKILL)
        except Exception: pass
    time.sleep(10)

def start_server(log_path):
    subprocess.run(['pkill', '-f', 'vllm serve'], check=False); time.sleep(5)     # 남은 서버가 GPU를 잡고 있으면 안 뜸
    log = open(log_path, 'w')
    proc = subprocess.Popen(
        ['vllm', 'serve', MODEL, '--revision', REVISION, '--served-model-name', 'qwen3-vl-4b',
         '--dtype', 'bfloat16', '--seed', '42', '--max-model-len', str(MAX_MODEL_LEN),
         '--gpu-memory-utilization', str(GPU_MEM_UTIL), '--port', str(PORT)],
        stdout=log, stderr=subprocess.STDOUT, start_new_session=True,
        env={**os.environ, 'VLLM_USE_V2_MODEL_RUNNER': '0'})
    for _ in range(720):                                 # 최대 60분 (첫 실행은 모델 다운로드 포함)
        try:
            if requests.get(f'http://localhost:{PORT}/v1/models', timeout=5).ok:
                return proc
        except Exception:
            pass
        if proc.poll() is not None:
            tail = Path(log_path).read_text(errors='ignore').splitlines()[-40:]
            raise RuntimeError('vLLM 서버가 종료됐습니다. 로그 마지막 줄:\n' + '\n'.join(tail))
        time.sleep(5)
    stop_server(proc)
    raise RuntimeError(f'서버 준비 시간 초과. 로그: {log_path}')

server = start_server(OUT_DIR / 'vllm_infer.log')
print('서버 준비 완료')

## 3. 추론
모델에는 **이미지 + 질문**만 보냅니다(정답, category, source_id는 보내지 않음). 이미 응답이 저장된 문항은 건너뜁니다.

In [ ]:
from concurrent.futures import ThreadPoolExecutor

PRED_PATH = OUT_DIR / ('predictions_cot.jsonl' if USE_COT else 'predictions.jsonl')
SUFFIX = ('\n\nThink step by step briefly, then finish with a final line in the form "Answer: <answer>".' if USE_COT
          else '\n\nLook at the music notation image. Reply with only the final answer in the form "Answer: <answer>".')

def ask(r):
    b64 = base64.b64encode((DATA_DIR / r['image']).read_bytes()).decode()
    body = {
        'model': 'qwen3-vl-4b', 'temperature': 0, 'seed': 42, 'max_tokens': MAX_NEW_TOKENS,
        'messages': [{'role': 'user', 'content': [
            {'type': 'image_url', 'image_url': {'url': f'data:image/png;base64,{b64}'}},
            {'type': 'text', 'text': r['question'] + SUFFIX}]}],
        'mm_processor_kwargs': {'min_pixels': MIN_PIXELS, 'max_pixels': MAX_PIXELS},
    }
    j = requests.post(f'http://localhost:{PORT}/v1/chat/completions', json=body, timeout=1800).json()
    ch = j['choices'][0]
    return {'id': r['id'], 'response': ch['message']['content'], 'finish_reason': ch['finish_reason'],
            'prompt_tokens': j['usage']['prompt_tokens']}

done = {json.loads(l)['id'] for l in open(PRED_PATH)} if PRED_PATH.exists() else set()
todo = [r for r in rows if r['id'] not in done]
print(f'추론 {len(todo)}문항 (이미 완료 {len(done)})')
t0 = time.time()
with ThreadPoolExecutor(CONCURRENCY) as ex, open(PRED_PATH, 'a', encoding='utf-8') as f:
    for out in ex.map(ask, todo):
        f.write(json.dumps(out, ensure_ascii=False) + '\n'); f.flush()
print(f'완료: {time.time() - t0:.0f}초')
stop_server(server)

## 4. 채점 및 결과
- **strict**: `Answer:` 뒤 최종 답이 정답과 정규화 후 완전히 같을 때
- **lenient**: 문장형으로 답해도 정답 라벨을 포함하면 인정 (예: "It is a half note." → Half note)
- 20문항이라 1문항이 5%p입니다. 카테고리별 수치는 참고용으로만 보세요.

In [ ]:
import pandas as pd

preds = {json.loads(l)['id']: json.loads(l) for l in open(PRED_PATH, encoding='utf-8')}
labels = scoring.label_sets(rows)
recs = []
for r in rows:
    p = preds[r['id']]
    s = scoring.score_one(r, p['response'], labels)
    recs.append({'id': r['id'], 'category': r['category'], 'gold': r['answer'], 'pred': s['pred'],
                 'strict': s['strict'], 'lenient': s['lenient'], 'finish': p['finish_reason'],
                 'prompt_tokens': p['prompt_tokens'], 'response': p['response']})
df = pd.DataFrame(recs)
df.to_csv(OUT_DIR / ('scores_cot.csv' if USE_COT else 'scores.csv'), index=False)

print(f"전체  strict {df.strict.sum()}/{len(df)} ({df.strict.mean():.0%}) | lenient {df.lenient.sum()}/{len(df)} ({df.lenient.mean():.0%})")
print(f"잘림(length) {int((df.finish == 'length').sum())}문항 | 프롬프트 토큰 중앙값 {int(df.prompt_tokens.median())}")
display(df.groupby('category')[['strict', 'lenient']].agg(['sum', 'count']))
display(df[['id', 'category', 'gold', 'pred', 'strict', 'lenient']])

### 틀린 문항의 모델 응답 원문

In [ ]:
for _, x in df[~df.lenient].iterrows():
    print(f"[{x.id}] ({x.category}) 정답: {x.gold} | 추출: {x.pred}\n  응답: {x.response[:300]!r}\n")